# FastMCP 서버 — 실습본

원본: `src/raw/Chapter02/01-mcp/01_mcp_server.py`  
슬라이드: `docs/raw/Chapter02/day01/3-2-model-context-protocol.pdf`

## 학습 목표

- Tool·Resource·Prompt와 stdio/SSE 실행
- 원본 코드를 CodeMate와 함께 읽고 실행 결과를 점검한다.


## 코드 흐름

FastMCP 생성 → 데코레이터 등록 → stdio 또는 SSE 서버 시작

**읽을 때 확인할 점:** _tasks는 메모리 목록이므로 서버를 재시작하면 사라진다. 함수 등록과 서버 실행을 구별한다.


## 실행 전 확인

Jupyter 실행 루프와 서버 실행이 충돌하지 않도록 setup 셀이 `__name__`을 설정한다. 원본 코드 셀은 기능을 등록하지만 서버를 시작하지 않는다. 저장소 루트의 별도 터미널에서 `uv run python src/raw/Chapter02/01-mcp/01_mcp_server.py`로 SSE 서버를 시작한 뒤 클라이언트를 연결한다. stdio는 원본 스크립트에 `--stdio`를 준다.

아래 원본 코드 셀은 실행을 직접 선택해야 하는 `manual-run` 셀입니다. 실행 준비 셀은 프로젝트 루트를 찾고 원본 `.py`의 경로를 `__file__`에 설정합니다.


In [ ]:
from pathlib import Path
import os
import sys

_repo_root = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "pyproject.toml").is_file() and (p / "Chapter02").is_dir()),
    None,
)
if _repo_root is None:
    raise FileNotFoundError("저장소 안에서 이 노트북을 열어 주세요.")
os.chdir(_repo_root)
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root / "src"))
__file__ = str(_repo_root / "src/raw/Chapter02/01-mcp/01_mcp_server.py")
# Jupyter에서는 스크립트 전용 __main__ 분기를 실행하지 않습니다.
__name__ = "notebook"


## 원본 코드

다음 셀에는 원본 `.py` 전체가 그대로 들어 있습니다.


In [ ]:
"""
MCP 서버 만들기 (FastMCP)

커스텀 MCP 서버를 직접 만들어 실행한다.
02_mcp_client.py가 이 서버에 연결해 도구를 사용한다.

실행:
  python 01_mcp_server.py            # HTTP/SSE 모드 (기본, 포트 8000)
  python 01_mcp_server.py --stdio    # stdio 모드 (Claude Code 등 MCP Host 연결용)

Claude Code에 등록하는 방법 (stdio):
  claude mcp add tutorial-server -- python /절대경로/01_mcp_server.py --stdio
"""

import sys
from datetime import datetime

from fastmcp import FastMCP

# FastMCP: Anthropic MCP SDK 위에서 동작하는 고수준 래퍼.
# TODO: @mcp.tool(), @mcp.resource(), @mcp.prompt() 데코레이터로 기능을 정의한다.
mcp = FastMCP('AI 강의 튜토리얼 서버')


# ── Tool: LLM이 호출할 수 있는 함수 ──────────────────────────────────────────
@mcp.tool()
def get_current_time() -> str:
    """현재 날짜와 시간을 반환한다."""
    return datetime.now().strftime('%Y년 %m월 %d일 %H:%M:%S')

@mcp.tool()
def calculate_bmi(weight_kg: float, height_cm: float) -> dict:
    """체중(kg)과 키(cm)로 BMI를 계산한다."""
    height_m = height_cm / 100
    bmi = weight_kg / (height_m ** 2)
    if bmi < 18.5:
        category = '저체중'
    elif bmi < 23:
        category = '정상'
    elif bmi < 25:
        category = '과체중'
    else:
        category = '비만'
    return {'bmi': round(bmi, 1), 'category': category}


_tasks: list[dict] = []

@mcp.tool()
def add_task(title: str, priority: int = 1) -> str:
    """할 일 목록에 새 항목을 추가한다. priority는 1(낮음)~3(높음)."""
    task = {'id': len(_tasks) + 1, 'title': title, 'priority': priority, 'done': False}
    _tasks.append(task)
    return f'할 일 추가됨: [{task["id"]}] {title} (우선순위: {priority})'

@mcp.tool()
def list_tasks() -> list[dict]:
    """현재 할 일 목록을 반환한다."""
    return _tasks if _tasks else [{'message': '할 일이 없습니다.'}]

@mcp.tool()
def complete_task(task_id: int) -> str:
    """할 일을 완료 처리한다."""
    for task in _tasks:
        if task['id'] == task_id:
            task['done'] = True
            return f'완료: {task["title"]}'
    return f'ID {task_id}를 찾을 수 없습니다.'


# ── Resource: LLM에게 제공할 데이터/문서 ──────────────────────────────────────
@mcp.resource('config://app-info')
def get_app_info() -> str:
    """이 MCP 서버의 기본 정보를 제공한다."""
    return """
    서버명: AI 강의 튜토리얼 MCP 서버
    버전: 1.0.0
    기능: 시간 조회, BMI 계산, 할 일 관리
    """


# ── Prompt: 자주 쓰는 프롬프트 템플릿 ────────────────────────────────────────
@mcp.prompt()
def task_summary_prompt(user_name: str) -> str:
    """할 일 목록 요약 요청 프롬프트를 생성한다."""
    return f'{user_name}님의 할 일 목록을 우선순위별로 정리하고 오늘 집중해야 할 항목을 추천해줘.'


# ── 실행 ──────────────────────────────────────────────────────────────────────

# TODO: MCP 서버 모드 정의
if __name__ == '__main__':
    if '--stdio' in sys.argv:
        # stdio 모드: Claude Code, Cursor 등 MCP Host와 직접 연결
        mcp.run(transport='stdio')
    else:
        print('MCP 서버 시작: http://localhost:8000/sse')
        print('종료하려면 Ctrl+C')
        # HTTP/SSE 모드: 02_mcp_client.py에서 localhost:8000으로 연결
        mcp.run(transport='sse', host='0.0.0.0', port=8000)        


## CodeMate와 함께 검토

> Tool, Resource, Prompt를 분류하고 BMI 도구의 입력·출력 예시와 서버 모드별 연결 방법을 설명해 줘. 원본은 바꾸지 말고 검토 내용을 먼저 말해 줘.

제안된 변경은 어떤 문제를 해결하는지, 어떤 결과로 확인할지 설명해 보세요.

## 확인 질문

1. Resource와 Tool의 차이는?
2. 서버 재시작 후 할 일 목록은?
